# Remove LLM ads that contain people-area annotations

This notebook creates a page-compatible derivative of the final LLM annotation JSONL. An advertisement is removed when its `groups` array contains at least one people-area annotation. Every retained advertisement must have an empty `groups` array and at least one record in `people`. The source file is never modified.

Outputs comprise the cleaned JSONL, an ad-level decision log, a metric/value statistics table, and a JSON provenance record.

In [ ]:
from __future__ import annotations

import csv
import hashlib
import json
import os
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

INPUT_ENV = "LLM_ANNOTATIONS_PATH"
OUTPUT_ENV = "LLM_ANNOTATIONS_CLEAN_OUTPUT_DIR"

input_candidates = [
    Path("../../data/annotations/llm-annotations/final.jsonl"),
    Path("../../../qwen_iteration/final_pipeline_standalone/output/production/full_pages_joined_v1/pages/assembled/final.jsonl"),
]
source_path = Path(os.environ[INPUT_ENV]) if os.environ.get(INPUT_ENV) else next(
    (path for path in input_candidates if path.exists()),
    input_candidates[0],
)
output_dir = Path(os.environ.get(OUTPUT_ENV, "../../data/processed/llm-annotations/individual-only"))

cleaned_path = output_dir / "llm-annotations-individual-ads-only.jsonl"
log_path = output_dir / "llm-annotations-people-area-ad-cleaning-log.csv"
stats_path = output_dir / "llm-annotations-people-area-ad-cleaning-stats.csv"
metadata_path = output_dir / "llm-annotations-people-area-ad-cleaning-metadata.json"

if not source_path.is_file():
    raise FileNotFoundError(
        f"LLM annotation source not found at {source_path}. Set {INPUT_ENV} to the source JSONL path."
    )
output_dir.mkdir(parents=True, exist_ok=True)

print(f"Source: {source_path}")
print(f"Output directory: {output_dir}")

In [ ]:
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def read_source_rows(path: Path):
    with path.open(encoding="utf-8") as handle:
        for line_number, line in enumerate(handle, start=1):
            if not line.strip():
                raise ValueError(f"Blank JSONL record at source line {line_number}")
            try:
                yield line_number, json.loads(line)
            except json.JSONDecodeError as error:
                raise ValueError(f"Invalid JSON at source line {line_number}: {error}") from error


log_fields = [
    "source_line_number",
    "image_id",
    "filename",
    "advertisement_id",
    "action",
    "reason",
    "people_area_count",
    "individual_annotation_count",
    "face_depiction_count_band",
    "has_outstanding_individuals",
]

In [ ]:
counts = Counter()
with cleaned_path.open("w", encoding="utf-8", newline="") as cleaned_handle, log_path.open(
    "w", encoding="utf-8", newline=""
) as log_handle:
    log_writer = csv.DictWriter(log_handle, fieldnames=log_fields, lineterminator="\n")
    log_writer.writeheader()

    for line_number, row in read_source_rows(source_path):
        counts["source_page_records"] += 1
        annotation = row.get("annotation")
        if not isinstance(annotation, dict):
            raise ValueError(f"Missing annotation object at source line {line_number}")
        advertisements = annotation.get("advertisements")
        if not isinstance(advertisements, list):
            raise ValueError(f"Missing advertisements array at source line {line_number}")

        kept_ads = []
        removed_on_page = 0
        for advertisement in advertisements:
            groups = advertisement.get("groups") or []
            people = advertisement.get("people") or []
            if not isinstance(groups, list) or not isinstance(people, list):
                raise ValueError(
                    f"Invalid people/groups arrays at line {line_number}, ad {advertisement.get('advertisement_id')}"
                )

            remove = len(groups) >= 1
            action = "removed" if remove else "retained"
            reason = "contains_people_area" if remove else "individual_annotations_only"
            counts["source_ad_records"] += 1
            counts[f"{action}_ad_records"] += 1
            counts["people_areas_removed"] += len(groups)
            if remove:
                removed_on_page += 1
                counts["individual_annotations_discarded_with_removed_ads"] += len(people)
            else:
                if not people:
                    raise ValueError(
                        f"Group-free ad without individual annotations at line {line_number}, "
                        f"ad {advertisement.get('advertisement_id')}"
                    )
                kept_ads.append(advertisement)
                counts["individual_annotations_remaining"] += len(people)

            log_writer.writerow(
                {
                    "source_line_number": line_number,
                    "image_id": row.get("image_id", ""),
                    "filename": row.get("filename", ""),
                    "advertisement_id": advertisement.get("advertisement_id", ""),
                    "action": action,
                    "reason": reason,
                    "people_area_count": len(groups),
                    "individual_annotation_count": len(people),
                    "face_depiction_count_band": advertisement.get("face_depiction_count_band", ""),
                    "has_outstanding_individuals": advertisement.get("has_outstanding_individuals", ""),
                }
            )

        annotation["advertisements"] = kept_ads
        retained_count = str(len(kept_ads))
        annotation["qualifying_ad_count"] = retained_count
        if isinstance(annotation.get("page"), dict):
            annotation["page"]["qualifying_ad_count"] = retained_count
        counts["pages_with_removed_ads"] += int(removed_on_page > 0)
        counts["pages_with_remaining_ads"] += int(bool(kept_ads))
        counts["pages_without_remaining_ads"] += int(not kept_ads)
        cleaned_handle.write(json.dumps(row, ensure_ascii=False, separators=(",", ":")) + "\n")

if counts["source_ad_records"] != counts["removed_ad_records"] + counts["retained_ad_records"]:
    raise AssertionError("Advertisement accounting does not reconcile")

removed_share = counts["removed_ad_records"] / counts["source_ad_records"]
stats_rows = [
    ("source_page_records", counts["source_page_records"], "page records", "Page-level JSONL records read from the source"),
    ("cleaned_page_records", counts["source_page_records"], "page records", "Page-level records written; the source page universe is preserved"),
    ("source_ad_records", counts["source_ad_records"], "advertisements", "Advertisements before filtering"),
    ("removed_ad_records", counts["removed_ad_records"], "advertisements", "Advertisements removed because groups contains at least one people area"),
    ("retained_ad_records", counts["retained_ad_records"], "advertisements", "Advertisements retained with groups empty and at least one individual annotation"),
    ("removed_ad_share", f"{removed_share:.8f}", "proportion", "Removed advertisements divided by source advertisements"),
    ("people_areas_removed", counts["people_areas_removed"], "people areas", "People-area annotations in removed advertisements"),
    ("individual_annotations_discarded_with_removed_ads", counts["individual_annotations_discarded_with_removed_ads"], "individual annotations", "Outstanding individual annotations discarded with people-area ads"),
    ("individual_annotations_remaining", counts["individual_annotations_remaining"], "individual annotations", "Individual annotations in retained advertisements"),
    ("pages_with_removed_ads", counts["pages_with_removed_ads"], "page records", "Pages from which at least one people-area ad was removed"),
    ("pages_with_remaining_ads", counts["pages_with_remaining_ads"], "page records", "Pages containing at least one retained advertisement"),
    ("pages_without_remaining_ads", counts["pages_without_remaining_ads"], "page records", "Pages whose cleaned advertisements array is empty"),
]
with stats_path.open("w", encoding="utf-8", newline="") as handle:
    writer = csv.writer(handle, lineterminator="\n")
    writer.writerow(["metric", "value", "unit", "definition"])
    writer.writerows(stats_rows)

print(f"Removed ads: {counts['removed_ad_records']:,}")
print(f"Retained ads: {counts['retained_ad_records']:,}")

In [ ]:
validated_pages = 0
validated_ads = 0
for line_number, row in read_source_rows(cleaned_path):
    validated_pages += 1
    advertisements = row["annotation"]["advertisements"]
    expected_count = str(len(advertisements))
    if row["annotation"]["qualifying_ad_count"] != expected_count:
        raise AssertionError(f"Top-level ad count mismatch at cleaned line {line_number}")
    if row["annotation"].get("page", {}).get("qualifying_ad_count") != expected_count:
        raise AssertionError(f"Page ad count mismatch at cleaned line {line_number}")
    for advertisement in advertisements:
        validated_ads += 1
        if advertisement.get("groups"):
            raise AssertionError(f"People area survived at cleaned line {line_number}")
        if not advertisement.get("people"):
            raise AssertionError(f"Retained ad lacks individual annotations at cleaned line {line_number}")

if validated_pages != counts["source_page_records"]:
    raise AssertionError("Cleaned page count does not match source page count")
if validated_ads != counts["retained_ad_records"]:
    raise AssertionError("Cleaned advertisement count does not match retained count")

metadata = {
    "schema_version": "llm_people_area_ad_cleaning_v1",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "source": {
        "file": source_path.name,
        "bytes": source_path.stat().st_size,
        "sha256": sha256_file(source_path),
    },
    "rule": {
        "remove_ad_when": "len(advertisement.groups) >= 1",
        "retain_ad_when": "advertisement.groups is empty and advertisement.people is non-empty",
        "page_rows_preserved": True,
        "count_fields_updated": ["annotation.qualifying_ad_count", "annotation.page.qualifying_ad_count"],
    },
    "counts": dict(counts),
    "outputs": {},
}
for path in (cleaned_path, log_path, stats_path):
    metadata["outputs"][path.name] = {
        "bytes": path.stat().st_size,
        "sha256": sha256_file(path),
    }
with metadata_path.open("w", encoding="utf-8") as handle:
    json.dump(metadata, handle, ensure_ascii=False, indent=2, sort_keys=True)
    handle.write("\n")

print("Validated outputs:")
for path in (cleaned_path, log_path, stats_path, metadata_path):
    print(f"- {path} ({path.stat().st_size:,} bytes)")